# Run Options

In [92]:
RUN_FRAGMENTATION = False
MOLECULE = 'CO2'
N_MODES = 4 # Number of Modes 
TRUNCATION = 4

# VQD Parameters:
num_states = 5 # Number of states to compute.
ndepth = 20 # Number of layers in the qumode circuit ansatz.
niter = 1 # Maximum number of iterations for the optimization.
threshold = 1e-3 # Convergence threshold for the optimization.
nfocks = [16, 16] # Designates the number of modes and number of fock states for each mode. For example, [16] means 1 mode with 16 fock states, and [16, 16] means 2 modes with 16 fock states each.

# Check for Necessary Files

In [41]:
from pathlib import Path
fragment_path = Path.cwd().parent / f'data/{MOLECULE}_Params.pkl' # if RUN_FRAGMENTATION, save to this location, else read fragments from this location 

In [42]:
if not RUN_FRAGMENTATION:
    if not fragment_path.exists():
        raise FileNotFoundError(f"Fragment file not found at {fragment_path}.\n"
                                "Please run the fragmentation script first by setting RUN_FRAGMENTATION to True.\n"
                                "Alternativley, download the files from https://github.com/sdonglab/QumVQD.git")

In [43]:
if MOLECULE == 'H2S':
    p = Path("..") / "data" / "H2S.hdf5"
elif MOLECULE == 'CO2':
    p = Path("..") / "data" / "CO2.hdf5"

if not p.exists():
    raise FileNotFoundError(f"Vibrational Hamiltonian file not found at  {p}.\n"
                            "Please download the file from <> and put it in the data directory.") # TODO: add link to download the file

# Prerequisite

In [44]:
from silence_tensorflow import silence_tensorflow
silence_tensorflow()

In [45]:
import tensorflow as tf

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

TensorFlow version: 2.21.0
GPU available: []


# Vibrational Hamiltonian

## prereqs

In [46]:
import numpy as np
import time

from third_party.vibrational_dynamics.util_hamil import operator_quad, hamil
from third_party.vibrational_dynamics.util_gfro import (
    norm_m,
    obtain_fragment,
    fragments,
    subtract,
    non_rotated_fragments,
    recreate_matrices,
    vectors,
    super_matrix_exp,
    extract_sub_matrices,
    tranformed_operators,
    compute_fragment,
    initial,
    flatten_matrices,
)
from third_party.vibrational_dynamics.util_qudratic import b_transform, recreate_quadH

from scipy.optimize import minimize



# Hamiltonian in the form of OpenFermion bosonic operator can be created with two functions
# "operator_quad" function creates hamiltonian generated with Gaussian
# "hamil" function creates hamiltonian provided by Xanadu

name = MOLECULE

# H = operator_quad(name) # Hamiltonian in OpenFermion bosonic operator form
# Options are "CO","CO2","NH3",
# model_hamiltonians options "D2","D3"

H = hamil(name)  # Hamiltonian in OpenFermion bosonic operator form
# Options are "H2S","CO2","CH4","H2S_3c","CO2_3c"
# "H2S_3c","CO2_3c" are with 3 mode coupling


n = N_MODES  # Number of modes

params = list()
norms = list()


# calculating the cost function(sum of square of the coeffiecients of H-H_f operator)


def cost(X):

    P, Q, G, L = recreate_matrices(X, n)  # Calculate P,Q,G,L from paramter vector X

    expA = super_matrix_exp(P, Q)  # Calculate exp([[P,Q],[Q,P]])

    U, V = extract_sub_matrices(
        expA, n
    )  # Extract U and V from exp([[P,Q],[Q,P]]) = [[U,-V],[-V,U]]

    C_b, C_bd = tranformed_operators(U, V, G)  # Calculate transformed operators

    H_f = compute_fragment(
        L, C_b, C_bd, n
    )  # Calculate quartic fragments from calculated L, C_b, C_bd

    H_vector, H_f_vector, sorted_ops = vectors(
        H, H_f
    )  # extract coeffiecients and corresponding operators from H and H_f

    # Finding indices corresponds to cubic and quartic terms in the Hamiltonian and the Fragment
    indices_cq = [i for i, op in enumerate(sorted_ops) if len(op) == 3 or len(op) == 4]

    cost_fn = []

    for i in indices_cq:
        value = H_vector[i] - H_f_vector[i]
        value = np.float64(value)
        value_sq = value**2
        cost_fn.append(value_sq)

    cost = np.real(np.sqrt((sum(cost_fn))))

    return cost


def printx(x):
    global iteration_count
    iteration_count += 1
    cost_value = cost(x)
    with open(f"{name}.txt", "a") as f:
        f.write(f"Step {iteration_count}: cost = {cost_value}\n")
    print(f"Step {iteration_count}: cost = {cost_value}")


current_norm = norm_m(H)
norms.append(current_norm)
print(" Norm of cubic and quartic terms")
print(f"\nInitial Norm : {current_norm}")


numtag = 4
tol = 1e-5
num_fragments = len(params)

options = {"maxiter": 1000, "disp": True} 


 Norm of cubic and quartic terms

Initial Norm : 161597.77026327705


## Run Fragmentation

In [47]:
Xs = []

In [48]:
if RUN_FRAGMENTATION:
  start_time = time.time()

  iteration_count = 0

  for k in range(numtag):

      # end decomposition if remaining norm is less than tol
      if current_norm < tol:
          print("Completed!!")
          break

      # obtain fragment

      P, Q, G, L = initial(n)
      X = 1e-3 * flatten_matrices(P, Q, G, L, n)
      Xs.append(X)

      result = minimize(
          cost, X, method="BFGS", tol=1e-7, options=options
      )  # , callback=printx)

      # obtain fragment from parameters in bosonic form
      H_f = fragments(result.x, n)

      # remove fragment from H
      H = subtract(H, H_f)

      current_norm = norm_m(H)

      # save current parameters and current norm
      params.append(result.x)

      norms.append(current_norm)

      num_fragments += 1

      print(
          f"fragment count : {num_fragments}, current norm : {np.round(current_norm, 7)}"
      )


  U, V, L, G, K = b_transform(H, n)  # params for the last fragment

  # Recreate the last fragment (qudratic)
  Hquad = recreate_quadH(U, V, L, G, K, n)

  end_time = time.time()
  elapsed_time = end_time - start_time
  print(f"Runtime : {elapsed_time}")


## Post-Processing

### Save to File

In [49]:
import pickle

In [50]:
if RUN_FRAGMENTATION:
  with open(fragment_path, "wb") as f:
      pickle.dump(params, f)

## import fragments, construct H_eff

In [51]:
import pickle

In [52]:
def sub_frags(H_target, H_frags):
  for i in range(len(H_frags)):
    H_target = subtract(H_target, H_frags[i])
  return H_target

In [76]:
from third_party.vibrational_dynamics.util_hamil import extract_coeffs_and_ops
from openfermion.ops import BosonOperator

# reconstruct bosonic operator from coefficients and operators
def reconstruct_boson_operator(coeffs, ops):
    boson_operator = BosonOperator()
    for coeff, op in zip(coeffs, ops):
        boson_operator += BosonOperator(op, coeff)
    return boson_operator


def H_quadatic_op(H):
    
    Hv,ops = extract_coeffs_and_ops(H)    

    indices_cq = [i for i, op in enumerate(ops) if len(op) < 3]

    Hvs = list()
    opss = list()

    for i in indices_cq:
        
        Hvs.append(Hv[i])
        opss.append(ops[i])
        
    H_qd = reconstruct_boson_operator(Hvs, opss)
    
    return H_qd

# Obtain fragments from previously calculated parameters in non-rotated coordinates
def non_rotated_fragments(name,H,n):
    # Read the JSON file
    file_path = f"params_{name}.json"
    with open(file_path, 'r') as file:
        data = json.load(file)

    # Convert the data into a list of NumPy arrays
    params = [np.array(entry) for entry in data]

    norms = list()

    norms.append(norm_m(H))  # norm of the initial H (cubic and quartic)
    
    # List to store each H2
    H_frags = []

    for i in range(len(params)):
        H2 = fragments(params[i],n)
        H_frags.append(H2)  # Save each H2 to the list
        H = subtract(H, H2)
        norms.append(norm_m(H))

    H_frags.insert(0,H_quadatic_op(H))
    #Edited by SVM 09/10/24. Original appended the upto quadratic term.
    #Now it is inserted in the 1st position.
    
    return H_frags,norms


In [95]:
# # code edited from non_rotated_fragments
# with open(fragment_path, 'rb') as file:
#     data = pickle.load(file)

# # Convert the data into a list of NumPy arrays
# params = [np.array(entry) for entry in data]

# norms = list()

# norms.append(norm_m(H))  # norm of the initial H (cubic and quartic)

# # List to store each H2
# H_frags = []

# for i in range(len(params)):
#     H2 = fragments(params[i],n)
#     H_frags.append(H2)  # Save each H2 to the list
#     H = subtract(H, H2)
#     norms.append(norm_m(H))

# H_frags.insert(0,H_quadatic_op(H))
# H_eff = sum(H_frags)

H_frags,norms = non_rotated_fragments('CO2',hamil('CO2'),n)
H_eff = sum(H_frags)
norms

[np.float64(161597.77026327705),
 np.float64(1050.8178571635565),
 np.float64(929.1492183046811)]

In [54]:
with open(fragment_path, "rb") as f:
  loaded_params = pickle.load(f)

H_frags = [fragments(param, n) for param in loaded_params]
H_0 = H_quadatic_op(H)  #sub_frags(H, H_frags)
H_eff = sum(H_frags) + H_0
H_frags.append(H_0)

In [55]:
len(loaded_params)

4

## Bosonic Op Conversion

In [56]:
trunc = TRUNCATION

In [57]:
from openfermion.linalg import boson_operator_sparse

H_frags_sparse = [boson_operator_sparse(H_frag, trunc=trunc) for H_frag in H_frags]

In [58]:
import tensorflow as tf

H_frags_tf = [tf.constant(H_frag.todense(), dtype=tf.complex128) for H_frag in H_frags_sparse]

## Diagonalization

### H_eff

In [88]:
H_eff_sparse = boson_operator_sparse(H_eff, trunc=trunc)

In [89]:
H_eff_eigenenergies = np.linalg.eigvalsh(H_eff_sparse.todense())

In [90]:
H_eff_eigenenergies[0:num_states]

array([-38487.95729948, -37138.90009035, -37138.87370284, -35787.93589098,
       -35117.21479212])

### H from file

In [61]:
H_original_sparse = boson_operator_sparse(H, trunc=4)
H_original_eigenenergies = np.linalg.eigvalsh(H_original_sparse.todense())
H_original_eigenenergies[:num_states]

array([2534.17615255, 3189.54465636, 3189.54465636, 3829.12675434,
       3847.16976539])

# Run QumVQD

In [62]:
from helpers.QumVQD_Helpers import *

KeyboardInterrupt: 

Exception ignored in: 'zmq.backend.cython._zmq.Frame.__dealloc__'
Traceback (most recent call last):
  File "zmq/backend/cython/_zmq.py", line 179, in zmq.backend.cython._zmq._check_rc
KeyboardInterrupt: 


In [ ]:
beta_factor = 4.0 * norm_m(H_eff)
H_eff_tf = [tf.constant(H_eff_sparse.todense(), dtype=tf.complex128)]
H_ops = H_eff_tf # Can use H_frags_tf or H_eff_tf
coeffs = np.ones(len(H_ops))

energies, state_params, results, psis = find_excited_states_multi_qumode(
    H_ops = H_ops, coeffs = coeffs, ndepth = ndepth,
    num_states = num_states, beta_factor=beta_factor,
    niter=niter, threshold=threshold, nfocks=nfocks, save_files=True, save_dir=f"output_data/{name}_VQD_results"
)

print("Energies found:", energies)

num_prev_states = 0

Finding state 0...
  [saved checkpoint to output_data/CO2_VQD_results after state 0]
State 0 energy: 4321.624166
Converged: False

Finding state 1...
Overlap with state 0: 1.563440e-04
  [saved checkpoint to output_data/CO2_VQD_results after state 1]
State 1 energy: 10277.330244
Converged: False

Finding state 2...
Overlap with state 0: 8.377400e-04
Overlap with state 1: 6.289912e-03
  [saved checkpoint to output_data/CO2_VQD_results after state 2]
State 2 energy: 10086.267755
Converged: False

Finding state 3...


In [ ]:
en = [float(energy) for energy in energies]
print(en)

[3302.5100600978453, 4563.81233475194, 5823.580119036882, 5825.986261653684, 5900.4247405279175]
